# core

> `astream` and `ClaudeRun`: stateless completions through the installed Claude Code

In [ ]:
#| default_exp core

`astream(msgs, ...)` returns a `ClaudeRun`: one stateless completion through the installed `claude`, using its login and subscription. The complete history (as `aidialog.msg_parts.Msg`s) is compiled into a native transcript via `fastclaude.session` under a fresh random session id and resumed. The caller owns the tool loop: tools are schemas, a run whose reply calls one ends at the `tool_use`, and a history ending in a `ToolResult` is continued by deferral: the pending call is marked in the transcript, the known result is held, and Claude collects it on resume via `fastclaude.protocol`. Iteration yields the raw stream-json events; `.messages` accumulates the canonical trace (signatures intact, tool names unqualified) and `.result` the terminal result, so the next request can replay everything. `run.interrupt()` ends a turn natively, and closing the stream reaps the process and removes the transcript. Runs default to an isolated XDG cache work dir, so no real project's sessions or settings are touched; pass `cwd=` for project context, and `native_tools=` to enable built-ins such as `'WebSearch'`.



In [ ]:
#| export
import asyncio, json, os, shutil, uuid
from contextlib import suppress
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.xdg import xdg_cache_home
from fastllm.anthropic import denorm_msgs, norm_parts
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult, Media
from fastclaude.session import *
from fastclaude.protocol import *

In [ ]:
from fastcore.test import *
import stat, sys, tempfile, textwrap

## The work dir

In Claude Code, the working directory is a storage key: the transcript we fabricate must sit in the `~/.claude/projects` folder derived from the directory the process runs in, or `--resume` finds nothing. It is also a behavior input, deciding which project settings and CLAUDE.md join the run, and whose session list our synthetic transcripts would pollute. So by default each run gets an isolated pseudo-project under the XDG cache dir: answers stop depending on where the host process happens to sit, no real project's sessions are touched, and everything filed under the cache path's project folder is ours to delete. Passing `cwd=` opts in to a real project's context instead.

In [ ]:
#| export
MCP_SERVER = 'fastclaude'
MCP_PREFIX = f'mcp__{MCP_SERVER}__'
SERVER_TOOLS = ('WebSearch','WebFetch')

def work_dir():
    "The default run directory: an isolated pseudo-project under the XDG cache dir"
    p = xdg_cache_home()/'fastclaude'
    p.mkdir(parents=True, exist_ok=True)
    return p

## Compiling history

Every call supplies the complete history as `aidialog.msg_parts.Msg` objects, so a caller can edit, hide, rewind, or replace anything before replaying it. Two endings are legal. A genuine user prompt (text or media) becomes the live turn, and everything before it becomes the transcript. A trailing tool result is the external loop's continuation: the transcript is filed through the pending `tool_use` and marked with `mk_deferred`, the known result is held for the bridge, and there is no live turn at all; the resume drives itself. Claude Code re-invokes only the last deferred call, so when several results return at once, all but the last are filed as ordinary records and exactly one defers. `denorm_msgs` converts the history to Anthropic-style wire messages, and `prefix_tools` qualifies callable-tool names the way this run offers them, leaving Claude Code's own tool names alone.


In [ ]:
#| export
def compile_msgs(
    msgs, # Complete history as `Msg`s, ending with a user prompt or with the tool results Claude asked for
):
    "`(history, prompt, deferred)`: wire messages to file, the live turn's content (None for a continuation), and the pending `(tool_use, tool_result)` pair"
    msgs = listify(msgs)
    if not msgs: raise ValueError('empty message history')
    den = prefix_tools(denorm_msgs(msgs), MCP_PREFIX, skip=SERVER_TOOLS)
    lc = den[-1]['content']
    if msgs[-1].role=='user' and any(isinstance(p, (Text,Media)) for p in msgs[-1].content): return den[:-1], lc, None
    if den[-1]['role']=='user' and isinstance(lc, list) and lc and all(b.get('type')=='tool_result' for b in lc):
        tus = {b['id']: b for m in den for b in (m['content'] if isinstance(m['content'], list) else []) if b.get('type')=='tool_use'}
        pend = [tus.get(b['tool_use_id']) for b in lc]
        if None in pend: raise ValueError('a trailing tool result answers no tool_use in the history')
        hist = den[:-1] + ([dict(role='user', content=lc[:-1])] if len(lc)>1 else [])
        return hist, None, (pend[-1], lc[-1])
    raise ValueError('history must end with a user prompt or tool results')

In [ ]:
prior = Msg('user', [Text('Measure the flux please.')])
call = Msg('assistant', [Text('Checking.'), ToolUse(id='t1', name='flux_meter', arguments={})])
result = Msg('tool', [ToolResult(id='t1', name='flux_meter', text='flux: 41.7 kf')])
prompt = Msg('user', [Text('And in gauss?')])
h = [prior,call,result,prompt]
h

`compile_msgs` files the first three messages as native history, qualifying its external tool name, and returns only the final user's content for the live turn.

In [ ]:
hist,prompt,deferred = compile_msgs(h)
test_eq((len(hist), deferred), (3, None))
test_eq(hist[1]['content'][1]['name'], 'mcp__fastclaude__flux_meter')
test_eq(prompt, [dict(type='text', text='And in gauss?')])
hist[1]

The same history cut at the tool result is the continuation. There is no live prompt; instead the pending pair comes back, the call qualified exactly as the transcript records it, ready for `mk_deferred` and the held reply:

In [ ]:
chist,cprompt,(ctu,ctr) = compile_msgs(h[:3])
test_eq((len(chist), cprompt), (2, None))
test_eq((ctu['id'], ctu['name']), ('t1', 'mcp__fastclaude__flux_meter'))
ctr

Several trailing results split by the re-invocation rule: all but the last file as an ordinary record in the history, and exactly the last defers.

In [ ]:
h2 = h[:1] + [Msg('assistant', [ToolUse(id='a', name='flux_meter', arguments={}), ToolUse(id='b', name='flux_meter', arguments=dict(unit='gauss'))]),
    Msg('tool', [ToolResult(id='a', name='flux_meter', text='flux: 41.7 kf'), ToolResult(id='b', name='flux_meter', text='flux: 41.7 gauss')])]
h2hist,_,h2d = compile_msgs(h2)
test_eq(h2hist[-1]['content'][0]['tool_use_id'], 'a')
test_eq(h2d[0]['id'], 'b')
h2hist[-1]

A history that ends neither way, is empty, or whose trailing result answers no recorded call is rejected before any process starts.

In [ ]:
with expect_fail(ValueError, contains='user prompt or tool results'): compile_msgs(h[:2])
with expect_fail(ValueError, contains='no tool_use'): compile_msgs(h[:1] + [Msg('tool', [ToolResult(id='zz', name='f', text='x')])])
with expect_fail(ValueError, contains='empty'): compile_msgs([])

## The command and environment

The executable is the user's installed `claude`, with their real config and login: that is the whole point, and it is why `CLAUDE_CONFIG_DIR` is never isolated. `--strict-mcp-config` is always passed, keeping their other configured MCP servers out of every run (without it, a run with no tools of its own will happily use whatever MCP servers the user has configured), `--tools` with an explicit list (empty by default) disables built-in tools until asked for, and the private SDK server entry in `--mcp-config` is how our in-process tools join. `--resume` uses the equals form so a dash-leading session id can never parse as a flag of its own. `setting_sources` picks which of the user's settings join the run: None keeps the CLI default (all of them), `['project']` loads only the project's, and `()` loads none. `mcp_config` adds external MCP servers, such as a stdio process, beside the private SDK entry. The environment blanks `ANTHROPIC_API_KEY`, so an inherited key cannot silently bill the API, and removes `CLAUDECODE`, so the child does not believe it is nested inside another Claude Code.

In [ ]:
#| export
def claude_cmd(
    model=None, # Model alias or full name; None uses the user's default
    resume=None, # Session id to resume, i.e. the transcript just written
    system=None, # System prompt; None keeps Claude Code's own
    tools=False, # Offer the private SDK MCP server?
    native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
    allowed=(), # `--allowedTools` entries, e.g. qualified callable names
    append_system=None, # Text appended to Claude Code's own system prompt, which stays
    setting_sources=None, # Settings that load, e.g. ['project']; () loads none; None keeps the CLI default (all)
    mcp_config=None, # Extra MCP server entries, e.g. `dict(clikernel=dict(type='stdio', command=...))`
    max_turns=None, # Bound on agent turns; None is unbounded
    max_budget=None, # Max USD for the run; None is unbounded
    permission_mode=None, # e.g. 'bypassPermissions'; None keeps the CLI default
    thinking=None, # 'adaptive' or 'disabled'; None keeps the CLI default
    effort=None, # Thinking effort: 'low', 'medium', or 'high'
    claude_path=None, # Explicit claude executable; found on PATH if None
):
    "argv for one headless stream-json claude run"
    c = [str(claude_path or shutil.which('claude') or 'claude'), '--output-format','stream-json',
        '--input-format','stream-json', '--verbose', '--include-partial-messages']
    if model: c += ['--model', model]
    if system is not None: c += ['--system-prompt', system]
    if append_system: c += ['--append-system-prompt', append_system]
    if resume: c += [f'--resume={resume}']
    servers = dict(mcp_config or {})
    if tools: servers[MCP_SERVER] = dict(type='sdk', name=MCP_SERVER)
    if servers: c += ['--mcp-config', json.dumps(dict(mcpServers=servers))]
    c.append('--strict-mcp-config')
    c += ['--tools', ','.join(native_tools)]
    if allowed: c += ['--allowedTools', ','.join(allowed)]
    if setting_sources is not None: c.append(f"--setting-sources={','.join(setting_sources)}")
    for f,v in (('--max-turns',max_turns), ('--max-budget-usd',max_budget), ('--permission-mode',permission_mode), ('--thinking',thinking), ('--effort',effort)):
        if v is not None: c += [f, str(v)]
    return c

def claude_env():
    "A child environment that cannot bill an API key and does not think it is nested"
    env = dict(os.environ, ANTHROPIC_API_KEY='')
    env.pop('CLAUDECODE', None)
    return env

In [ ]:
c = claude_cmd('sonnet', resume='-abc', tools=True, allowed=['mcp__fastclaude__flux_meter'])
test('--resume=-abc', c, in_)
test_eq(c[c.index('--tools')+1], '')
test('--strict-mcp-config', c, in_)
test('--strict-mcp-config', claude_cmd('sonnet'), in_)
test_eq(json.loads(c[c.index('--mcp-config')+1]), dict(mcpServers=dict(fastclaude=dict(type='sdk', name='fastclaude'))))
env = claude_env()
test_eq(env['ANTHROPIC_API_KEY'], '')
assert 'CLAUDECODE' not in env
c[1:]

The run-shaping options render only when given, so the default command stays minimal:

In [ ]:
c2 = claude_cmd(mcp_config=dict(clikernel=dict(type='stdio', command='clikernel-mcp')), setting_sources=['project'], max_turns=8)
test_eq(json.loads(c2[c2.index('--mcp-config')+1])['mcpServers']['clikernel']['type'], 'stdio')
test('--setting-sources=project', c2, in_)
test_eq(c2[c2.index('--max-turns')+1], '8')
assert '--max-turns' not in claude_cmd()
c2[1:]

## The run

A `ClaudeRun` is request-scoped: it owns exactly one temporary transcript and one process, then becomes terminal. Iterating it drives the whole lifecycle: compile and file the history under a fresh random session id, spawn `claude` resuming it, shake hands, send the live turn (a continuation sends none), and yield every non-control event raw. Along the way it accumulates the canonical trace: each full `assistant` event becomes a `Msg` via `norm_parts` (signatures intact, callable names unqualified), each tool-result `user` event a `tool` role `Msg`, and the terminal `result` lands on `.result`. A reply that calls one of the advertised tools simply ends there: the trace's last message carries the `ToolUse`, the caller executes it, and the next request's trailing `ToolResult` continues the loop.



In [ ]:
#| export
class ClaudeRun:
    "One stateless completion: a fresh transcript, one claude process, streamed events, a canonical trace"
    @delegates(claude_cmd, but=['model','resume','tools','native_tools','allowed'])
    def __init__(self,
        msgs, # Complete history as `Msg`s, ending with a user prompt or the tool results Claude asked for
        model='sonnet', # Model alias or full name
        tools=None, # Tool schemas to advertise, in either `tool_spec` form; the caller executes
        cwd=None, # Project directory for the run; the isolated `work_dir()` if None
        native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
        allowed=(), # Extra `--allowedTools` entries beyond the advertised schemas
        env=None, # Extra child environment variables, merged over `claude_env()`
        **kwargs, # Passed to `claude_cmd`, e.g. `system`, `setting_sources`, `mcp_config`
    ):
        store_attr('msgs,model,tools,cwd,native_tools,allowed,env')
        self.cmd_kwargs = kwargs
        self.messages,self.result,self.proc,self.proto = [],None,None,None
        self._names,self._closed,self._spath,self._deferred = {},False,None,None

    def __aiter__(self):
        if not hasattr(self, '_it'): self._it = self._run()
        return self._it

@delegates(ClaudeRun)
def astream(msgs, **kwargs):
    "Start one stateless completion; iterate the returned `ClaudeRun` for its raw events"
    return ClaudeRun(msgs, **kwargs)

Spawning files the history and builds the process. A fresh random session id avoids collisions when identical histories run concurrently; prompt caching depends on request content, not session reuse. A continuation appends the forged deferral record and hands the peer its one held reply. An empty history (a bare first prompt) writes no transcript and resumes nothing:



In [ ]:
#| export
@patch
async def _spawn(self:ClaudeRun):
    "Compile and file the history, then start Claude resuming it; returns the live turn's content (None for a continuation)"
    self.cwd = Path(self.cwd).expanduser() if self.cwd else work_dir()
    hist,prompt,deferred = compile_msgs(self.msgs)
    sid = str(uuid.uuid4())
    recs = msgs2recs(hist, key=sid, cwd=self.cwd)
    held = None
    if deferred:
        tu,tr = deferred
        recs.append(mk_deferred(tu, cwd=self.cwd))
        held,self._deferred = tool_reply(tr.get('content',''), tr.get('is_error', False)),tu['id']
    if recs:
        save_sess(recs, sid, self.cwd)
        self._spath = sess_file(sid, self.cwd)
    schemas = mk_tools(self.tools or [])
    allowed = [MCP_PREFIX+s['name'] for s in schemas] + list(self.native_tools) + list(self.allowed)
    argv = claude_cmd(self.model, resume=sid if recs else None, tools=bool(schemas),
        native_tools=self.native_tools, allowed=allowed, **self.cmd_kwargs)
    self.proc = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=self.cwd, env=dict(claude_env(), **(self.env or {})))
    self.proto = ClaudeProto(self.proc, tools=self.tools, held=held, server=MCP_SERVER)
    return prompt

Trace accumulation reads only full message events, ignoring partials: they are transcript-grade, one event per content block. Callable names are unqualified on the way back, so callers see the tool they registered rather than Claude's `mcp__` spelling. On a continuation, the re-invoked call's result event carries the result the caller itself supplied, so it is not folded back into the trace. Native tool results (such as `WebSearch`'s) and everything else still stream through raw:



In [ ]:
#| export
def unqual(nm):
    "The bare tool name for a possibly `mcp__fastclaude__`-qualified `nm`"
    return nm[len(MCP_PREFIX):] if nm and nm.startswith(MCP_PREFIX) else nm

def _flat(c): return c if isinstance(c, str) else '\n'.join(b.get('text','') for b in c if b.get('type')=='text')

@patch
def _track(self:ClaudeRun, m):
    "Fold one raw event into `.messages` and `.result`"
    t,c = m.get('type'), nested_idx(m, 'message', 'content')
    if t=='assistant' and isinstance(c, list):
        parts = norm_parts(m['message'])
        for p in parts:
            if isinstance(p, ToolUse): p.name = self._names[p.id] = unqual(p.name)
        self.messages.append(Msg('assistant', parts))
    elif t=='user' and isinstance(c, list) and c and all(b.get('type')=='tool_result' for b in c):
        if any(b.get('tool_use_id')==self._deferred for b in c): return
        self.messages.append(Msg('tool', [ToolResult(id=b.get('tool_use_id'), name=self._names.get(b.get('tool_use_id')),
            text=_flat(b.get('content',''))) for b in c]))
    elif t=='result': self.result = m

`_kick` performs the control handshake and sends the live user turn, if there is one: a continuation drives itself from the transcript. The drive starts it as a task because the same protocol event loop must already be consuming the handshake response.

In [ ]:
#| export
@patch
async def _kick(self:ClaudeRun, prompt):
    "Handshake, then the live user turn; a continuation sends none"
    await self.proto.initialize()
    if prompt is not None: await self.proto.send(dict(type='user', message=dict(role='user', content=prompt)))

The drive: events stream through `_track` to the caller, and the terminal result ends the run. However the loop ends, by result, consumer break, or cancellation, `aclose` runs:

In [ ]:
#| export
@patch
async def _run(self:ClaudeRun):
    "The event stream: spawn, kick off, yield raw events until the terminal result"
    prompt = await self._spawn()
    kick = asyncio.create_task(self._kick(prompt))
    try:
        async for m in self.proto.events():
            self._track(m)
            yield m
            if m.get('type')=='result': break
    finally:
        kick.cancel()
        await self.aclose()

`interrupt` delegates to Claude's native control request. The same event stream remains open to drain the aborted tail, whose terminal result reports the interruption.

In [ ]:
#| export
@patch
async def interrupt(self:ClaudeRun, timeout=30):
    "Claude's native interrupt: end the current turn; the stream stays open to drain the aborted tail"
    return await self.proto.interrupt(timeout)

Cleanup is the part that must work under cancellation, because a host's ctrl-C arrives as exactly that: the consumer task is cancelled, the stream is closed, and this sequence is all that stands between an abandoned turn and an orphaned process. Nothing is read back from the process, since a closed run's trace is never consulted.

Closing stdin asks Claude to finish; then wait, and escalate: terminate, then kill, each with a deadline.

`_wait_process` turns a bounded process wait into a boolean, so escalation can be expressed as a flat sequence rather than nested timeout handlers.

In [ ]:
#| export
async def _wait_process(proc, timeout=5):
    try:
        await asyncio.wait_for(proc.wait(), timeout)
        return True
    except (TimeoutError, asyncio.TimeoutError): return False

`_reap_process` first offers a normal wait, then termination, then killing. Each successful stage returns immediately; only the final kill is unbounded because the process must not survive cleanup.

In [ ]:
#| export
async def _reap_process(proc):
    if await _wait_process(proc): return
    with suppress(ProcessLookupError): proc.terminate()
    if await _wait_process(proc): return
    with suppress(ProcessLookupError): proc.kill()
    with suppress(Exception): await proc.wait()

Finally remove the exact transcript this run wrote, success or failure.

In [ ]:
#| export
@patch
async def _cleanup(self:ClaudeRun):
    p = self.proc
    if p and p.returncode is None:
        with suppress(Exception): p.stdin.close()
        await _reap_process(p)
    if self.proto: await self.proto.aclose()
    if self._spath: Path(self._spath).unlink(missing_ok=True)

The whole body runs shielded, so a second cancellation cannot abort it partway:

In [ ]:
#| export
@patch
async def aclose(self:ClaudeRun):
    "Close stdin, reap the process, and remove the transcript; idempotent and cancellation-shielded"
    if self._closed: return
    self._closed = True
    await asyncio.shield(asyncio.create_task(self._cleanup()))

## A scripted run

The lifecycle runs end to end against a scripted stand-in for `claude`, so the run's own logic - kickoff ordering, trace accumulation, terminal handling, transcript cleanup - is verified offline. The script answers any control request, echoes the user turn as an assistant message, and finishes with a result:

In [ ]:
# chkstyle: ignore-node
fake_src = textwrap.dedent('''
    #!/usr/bin/env python3
    import sys, json
    def w(o): sys.stdout.write(json.dumps(o)+'\\n'); sys.stdout.flush()
    for line in sys.stdin:
        m = json.loads(line)
        if m.get('type')=='control_request':
            w(dict(type='control_response', response=dict(subtype='success', request_id=m['request_id'], response={})))
        elif m.get('type')=='user':
            c = m['message']['content']
            txt = 'echo: '+(c if isinstance(c, str) else c[0].get('text',''))
            w(dict(type='assistant', message=dict(role='assistant', content=[dict(type='text', text=txt)])))
            w(dict(type='result', subtype='success', result=txt))
    ''').strip()

Writing that peer as an executable preserves the real subprocess, pipe, handshake, and cleanup boundaries while keeping the lesson deterministic and model-free.

In [ ]:
fake_cc = Path(tempfile.mkdtemp())/'claude'
fake_cc.write_text(fake_src+'\n')
fake_cc.chmod(fake_cc.stat().st_mode | stat.S_IXUSR)

In [ ]:
scratch = Path(tempfile.mkdtemp())
run = astream(h, claude_path=fake_cc, cwd=scratch)
got = [m async for m in run]
test_eq(run.result['result'], 'echo: And in gauss?')
test_eq([m.role for m in run.messages], ['assistant'])
test_eq(run.messages[0].content[0].text, 'echo: And in gauss?')
test_eq(run._spath.exists(), False)
[m['type'] for m in got]

## Live runs

The examples below use the real authenticated CLI (they spend tokens, so stay out of automated runs). First the stop: Claude decides to use the advertised tool, the defer hook ends the turn, and the run finishes with the pending `ToolUse` as the last part of its trace. `flux_meter` is passed as a tool, but only its schema travels: the body does not run here, and nothing has answered the call yet:



In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

[('assistant', ['Thinking']), ('assistant', ['ToolUse'])]

`flux_meter` is advertised as a schema but never invoked by FastClaude. Iterating the run consumes one whole process, which ends at the external tool boundary.

In [ ]:
#| eval: false
lmsgs = [Msg('user', [Text('Use the flux_meter tool with unit="gauss", then reply with exactly the tool output.')])]
lrun = astream(lmsgs, model='claude-sonnet-5', tools=[flux_meter])
levs = [m async for m in lrun]
L(m.get('type') for m in levs).unique()

The accumulated trace ends with Claude's unqualified `ToolUse`, and the process has already exited with a successful result. Nothing is left running.

In [ ]:
#| eval: false
ltu = lrun.messages[-1].content[-1]
test_eq((type(ltu), ltu.name, ltu.arguments), (ToolUse, 'flux_meter', dict(unit='gauss')))
test_eq(lrun.result['subtype'], 'success')
[(m.role, [type(p).__name__ for p in m.content]) for m in lrun.messages]

The caller executes and continues: run the requested call (here, our own function), append its result to the history, and `astream` again. The trailing `ToolResult` becomes the deferral. No live prompt is sent, Claude collects the held result on resume, and the continuation arrives as this run's trace, with the supplied result not folded back in. Nothing re-executes:



In [ ]:
#| eval: false
out = await flux_meter(**ltu.arguments)
cmsgs = lmsgs + lrun.messages + [Msg('tool', [ToolResult(id=ltu.id, name=ltu.name, text=out)])]
crun = astream(cmsgs, model='claude-sonnet-5', tools=[flux_meter])
cevs = [m async for m in crun]
test_eq([m.role for m in crun.messages], ['assistant'])
crun.result['result']


'flux: 41.7 gauss'

Interruption still matters mid-generation: `run.interrupt()` sends the native control request, the stream stays open to drain the aborted tail, and the terminal result reports the interruption distinctly from a provider failure. Mid-tool interruption is no longer this library's concern, because tools run in the caller, who cancels its own work:

In [ ]:
#| eval: false
irun = astream([Msg('user', [Text('Write a 2000 word essay on the history of magnetometry. Do not stop early.')])])
n = 0
async for m in irun:
    if m.get('type')=='stream_event' and (n := n+1)==20: asyncio.ensure_future(irun.interrupt())
test_eq([m.role for m in irun.messages], ['assistant'])
{k: irun.result.get(k) for k in ('subtype','is_error')}


{'subtype': 'error_during_execution', 'is_error': True}

Statelessness closes the loop: the finished exchange replays as ordinary history under a new prompt. The signed thinking block replays into the transcript, the tool exchange is there to be referred to, and nothing re-executes:



In [ ]:
#| eval: false
qmsgs = cmsgs + crun.messages + [Msg('user', [Text('What unit did the flux_meter report in? One word only.')])]
qrun = astream(qmsgs, model='claude-sonnet-5', tools=[flux_meter])
qevs = [m async for m in qrun]
qrun.result['result']


'Gauss'

## Cleanup

Runs remove their own transcripts, so only empty per-run project folders remain; the scripted run's scratch folder is ours to delete, and the shared cache-dir folder is scratch by contract.

In [ ]:
shutil.rmtree(sess_dir(scratch), ignore_errors=True)
shutil.rmtree(sess_dir(work_dir()), ignore_errors=True)
shutil.rmtree(fake_cc.parent, ignore_errors=True)
shutil.rmtree(scratch, ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()